# CatchCam on Colab (fast processing on a free GPU)

**Before you start**
1. In Colab: *Runtime → Change runtime type → T4 GPU*.
2. In Google Drive make a folder **CatchCamColab** with:
   - `catchcam_project.zip` (made on your PC with `python make_colab_bundle.py`)
   - a folder `clips` with your short clips named after the cameras: `cam1.mp4`, `cam2.mp4`, ...
3. Edit the **START TIMES** in step 4 to the clock shown on each camera at the first frame of its clip.

Then run the cells one after another.

In [ ]:
#@title 1. Check the GPU
!nvidia-smi -L || echo 'No GPU: Runtime > Change runtime type > T4 GPU'

In [ ]:
#@title 2. Connect Google Drive
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/CatchCamColab'
!ls -la $BASE $BASE/clips

In [ ]:
#@title 3. Install CatchCam
!rm -rf /content/catchcam && mkdir /content/catchcam
!unzip -oq $BASE/catchcam_project.zip -d /content/catchcam
!pip -q install ultralytics networkx reportlab
%cd /content/catchcam
!ls

In [ ]:
# If the output says "Skipped ...: <name> is not installed", run   !pip install <name>   in a new cell and run this cell again.
#@title 4. Process the clips (about 1-3 minutes per camera on a T4)
START_TIMES = {          # clock shown on the camera at the first frame of the clip
    'cam1': '12:38:53',
    'cam2': '12:39:29',
    'cam3': '12:42:03',
    'cam4': '12:44:15',
}
WEIGHTS = 'yolov8m.pt'   # more accurate than the laptop default; fine on a GPU
starts = ' '.join(f'--start {c}={t}' for c, t in START_TIMES.items())
!python process_clips.py $BASE/clips --data /content/demo_data --weights $WEIGHTS --speed normal $starts

In [ ]:
#@title 5. Save the results to Drive and download them
!cp /content/demo_data.zip $BASE/
from google.colab import files
files.download('/content/demo_data.zip')

## On your PC
1. Unzip `demo_data.zip` into `D:\CatchCam` (you get `D:\CatchCam\demo_data`).
2. In PowerShell:
```
cd D:\CatchCam
.\venv\Scripts\activate
$env:CATCHCAM_DATA_DIR = "D:\CatchCam\demo_data"
python -m uvicorn api:app --port 8000
```
3. Open http://localhost:8000/ . Your old cameras are untouched (they stay in the normal data folder).